# SARIMA Grid Search

## Purpose
Conducts an exhaustive grid search over SARIMA `(p, d, q)(P, D, Q, 12)` parameter combinations to identify the configuration that minimises walk-forward RMSE on the training dataset.

`SARIMAX` handles seasonal differencing internally; no manual pre-differencing is applied. The search is run twice: first over a broad unrestricted grid, then with `D` fixed to `1` to concentrate on configurations with explicit seasonal integration, which is appropriate given the strong seasonal structure in this series.

## Inputs
- `data/dataset.csv` — Training dataset (93 monthly observations)

## Outputs
- The RMSE for each evaluated configuration and the overall best configuration from each search pass.

In [1]:
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
from math import sqrt
import warnings
warnings.filterwarnings('ignore')

## Single-Order Evaluation Function

Runs walk-forward validation for a given `(p, d, q)(P, D, Q, 12)` order pair and returns the RMSE. `enforce_stationarity=False` and `enforce_invertibility=False` are set to allow the optimiser to explore the full parameter space without hard rejections.

In [2]:
# evaluate a SARIMA model for given orders and return RMSE
def evaluate_sarima_model(X, order, seasonal_order):
    X = X.astype('float64')
    train_size = int(len(X) * 0.50)
    train, test = X[0:train_size], X[train_size:]
    history = [x for x in train]
    predictions = list()
    for t in range(len(test)):
        model = SARIMAX(history, order=order, seasonal_order=seasonal_order,
                        enforce_stationarity=False, enforce_invertibility=False)
        model_fit = model.fit(disp=False)
        # Guard: reject fits where MLE optimisation failed to converge
        if not model_fit.mle_retvals['converged']:
            raise RuntimeError(f'SARIMA{order}x{seasonal_order} did not converge at step {t}')
        yhat = model_fit.forecast()[0]
        predictions.append(yhat)
        history.append(test[t])
    rmse = sqrt(mean_squared_error(test, predictions))
    return rmse

## Grid Search Driver

Iterates over every combination of the supplied `p`, `d`, `q`, `P`, `D`, `Q` ranges and calls `evaluate_sarima_model` for each. Configurations that raise any exception are skipped via `except: continue` and the configuration with the lowest RMSE is tracked throughout.

The grid is deliberately conservative — `p, q ∈ [0, 2]` and `P, Q ∈ [0, 2]` — because the short series (93 observations) cannot reliably identify high-order seasonal models and convergence failures increase sharply with model complexity.

In [3]:
# evaluate combinations of (p,d,q)(P,D,Q,12)
# Grid is deliberately conservative: short series (93 obs) cannot reliably
# identify high-order seasonal models and convergence failures increase sharply.
def evaluate_models(dataset, p_values, d_values, q_values,
                    P_values, D_values, Q_values, m=12):
    dataset = dataset.astype('float32')
    best_score, best_cfg = float('inf'), None
    for p in p_values:
        for d in d_values:
            for q in q_values:
                for P in P_values:
                    for D in D_values:
                        for Q in Q_values:
                            order = (p, d, q)
                            seasonal_order = (P, D, Q, m)
                            try:
                                rmse = evaluate_sarima_model(dataset, order, seasonal_order)
                                if rmse < best_score:
                                    best_score, best_cfg = rmse, (order, seasonal_order)
                                print('SARIMA%s x %s RMSE=%.3f' % (order, seasonal_order, rmse))
                            except:
                                continue
    print('Best SARIMA%s x %s RMSE=%.3f' % (best_cfg[0], best_cfg[1], best_score))

## Load Training Data

In [4]:
series = pd.read_csv('data/dataset.csv', index_col=0, parse_dates=True).iloc[:, 0]
series.head()

Month
1964-01-01    2815
1964-02-01    2672
1964-03-01    2755
1964-04-01    2721
1964-05-01    2946
Name: Sales, dtype: int64

## Execute Grid Search — Pass 1 (Unrestricted)

Search over `p ∈ [0, 2]`, `d ∈ [0, 1]`, `q ∈ [0, 2]`, `P ∈ [0, 2]`, `D ∈ [0, 1]`, `Q ∈ [0, 2]` with `m=12`. Runtime will be several minutes due to the walk-forward re-fitting at every step for every candidate configuration.

In [5]:
# evaluate parameters
# Non-seasonal: p in [0,1,2], d in [0,1], q in [0,1,2]
# Seasonal:     P in [0,1,2], D in [0,1], Q in [0,1,2], m=12
# Note: this grid runs 47 walk-forward fits per candidate — expect several minutes.
p_values = range(0, 3)
d_values = range(0, 2)
q_values = range(0, 3)
P_values = range(0, 3)
D_values = range(0, 2)
Q_values = range(0, 3)
evaluate_models(series.values, p_values, d_values, q_values,
                P_values, D_values, Q_values)

SARIMA(0, 0, 0) x (0, 0, 0, 12) RMSE=6057.100
SARIMA(0, 0, 0) x (0, 1, 0, 12) RMSE=943.061
SARIMA(0, 0, 0) x (1, 0, 0, 12) RMSE=972.575
SARIMA(0, 0, 0) x (1, 1, 0, 12) RMSE=1020.426
SARIMA(0, 0, 0) x (2, 1, 0, 12) RMSE=1009.688
SARIMA(0, 0, 1) x (0, 0, 0, 12) RMSE=4058.657
SARIMA(0, 0, 1) x (0, 1, 0, 12) RMSE=941.420
SARIMA(0, 0, 1) x (1, 1, 0, 12) RMSE=1064.877
SARIMA(0, 0, 1) x (2, 1, 0, 12) RMSE=1037.026
SARIMA(0, 0, 2) x (0, 0, 0, 12) RMSE=3842.113
SARIMA(0, 0, 2) x (0, 1, 0, 12) RMSE=959.377
SARIMA(0, 0, 2) x (1, 1, 0, 12) RMSE=1075.644
SARIMA(0, 1, 0) x (0, 0, 0, 12) RMSE=3186.501
SARIMA(0, 1, 0) x (0, 1, 0, 12) RMSE=1145.923
SARIMA(0, 1, 0) x (1, 1, 0, 12) RMSE=1060.994
SARIMA(0, 1, 0) x (2, 1, 0, 12) RMSE=1168.602
SARIMA(0, 1, 1) x (0, 0, 0, 12) RMSE=3267.756
SARIMA(0, 1, 1) x (0, 1, 0, 12) RMSE=955.970
SARIMA(0, 1, 1) x (1, 0, 0, 12) RMSE=978.114
SARIMA(0, 1, 1) x (1, 1, 0, 12) RMSE=931.004
SARIMA(0, 1, 1) x (2, 0, 0, 12) RMSE=899.683
SARIMA(0, 1, 1) x (2, 1, 0, 12) RMSE=988.3

## Execute Grid Search — Pass 2 (D Fixed to 1)

Constrains `D=1` to enforce seasonal integration, consistent with the strong 12-month periodicity present in the data. Non-seasonal `d` is still allowed to vary. This pass narrows the search to the most practically relevant configurations and typically identifies a more stable best model.

In [6]:
D_values = [1]   # fix seasonal differencing to 1
P_values = range(0, 3)
Q_values = range(0, 3)
d_values = range(0, 2)  # non-seasonal d can still vary
evaluate_models(series.values, p_values, d_values, q_values,
                P_values, D_values, Q_values)

SARIMA(0, 0, 0) x (0, 1, 0, 12) RMSE=943.061
SARIMA(0, 0, 0) x (1, 1, 0, 12) RMSE=1020.426
SARIMA(0, 0, 0) x (2, 1, 0, 12) RMSE=1009.688
SARIMA(0, 0, 1) x (0, 1, 0, 12) RMSE=941.420
SARIMA(0, 0, 1) x (1, 1, 0, 12) RMSE=1064.877
SARIMA(0, 0, 1) x (2, 1, 0, 12) RMSE=1037.026
SARIMA(0, 0, 2) x (0, 1, 0, 12) RMSE=959.377
SARIMA(0, 0, 2) x (1, 1, 0, 12) RMSE=1075.644
SARIMA(0, 1, 0) x (0, 1, 0, 12) RMSE=1145.923
SARIMA(0, 1, 0) x (1, 1, 0, 12) RMSE=1060.994
SARIMA(0, 1, 0) x (2, 1, 0, 12) RMSE=1168.602
SARIMA(0, 1, 1) x (0, 1, 0, 12) RMSE=955.970
SARIMA(0, 1, 1) x (1, 1, 0, 12) RMSE=931.004
SARIMA(0, 1, 1) x (2, 1, 0, 12) RMSE=988.342
SARIMA(0, 1, 2) x (0, 1, 0, 12) RMSE=968.595
SARIMA(0, 1, 2) x (1, 1, 0, 12) RMSE=931.500
SARIMA(0, 1, 2) x (2, 1, 0, 12) RMSE=1030.791
SARIMA(1, 0, 0) x (0, 1, 0, 12) RMSE=945.970
SARIMA(1, 0, 0) x (1, 1, 0, 12) RMSE=1036.167
SARIMA(1, 0, 0) x (2, 1, 0, 12) RMSE=957.265
SARIMA(1, 0, 1) x (0, 1, 0, 12) RMSE=937.543
SARIMA(1, 0, 1) x (1, 1, 0, 12) RMSE=1002.331